# 线性规划（LP）：一个模型一个 Notebook

## 这个模型解决什么问题
资源有限、目标单一、关系全是线性的情况下，决定「各做多少」。
典型场景：生产计划、配料、排产、预算分配、人员排班（连续量）。

标准形式（本 Notebook 统一按这个写，最大化最小化只差一个符号）：

```
max  c'x
s.t. A_ub · x <= b_ub
     lb <= x <= ub
```

## 两种数据输入方式

| 方式 | 什么时候用 | 用哪个函数 |
| --- | --- | --- |
| **原始表** | 附件还是长表/宽表，有脏数据（空格、999、文本数字、重复行） | `build_lp_from_table` + `clean_frame` |
| **已处理矩阵** | 系数已经整理好（自己算完，或来自 `数据学习.ipynb` 的清洗结果） | `solve_lp` |

## 目录
- 0 环境与路径
- 1 配置区（换题目只改这里）
- 2 数据接口（读文件 + 清洗 + 长表转宽表）
- 3 建模与求解（`LpSpec` / `build_lp_from_table` / `solve_lp` / 约束核对）
- 4 示例 A：原始脏表 → LP
- 5 示例 B：已处理矩阵 → LP
- 6 结果导出与画图

> 依赖：`numpy pandas scipy`，画图要 `matplotlib`，导出 xlsx 要 `openpyxl`（Python 3.11 实测）。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区：换题目只改这个字典
# =====================================================================
CFG = dict(
    # ---- 输入 ----
    data_path=DATA_DIR / "优化_生产计划_原始.csv",  # 原始表路径；换题目改这里
    sheet_name=0,                                   # 读 xlsx 时指定工作表
    # ---- 模型方向 ----
    maximize=True,          # True=最大化利润/收益；False=最小化成本
    # ---- 输出 ----
    tag="生产计划",          # 输出目录名：输出数据/优化_生产计划/
    make_plots=True,        # False 时一张图都不画（批量跑时省时间）
    save_excel=True,        # False 时只写 CSV
)
print("配置：", CFG)


配置： {'data_path': WindowsPath('D:/git项目/导入数据/优化_生产计划_原始.csv'), 'sheet_name': 0, 'maximize': True, 'tag': '生产计划', 'make_plots': True, 'save_excel': True}


## 2. 数据接口

读文件、清洗、长表转宽表都在这里。**数据已经处理好时可以跳过本节**，直接用第 5 节的矩阵入口。


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 建模与求解

`LpSpec` 负责「表 → 系数」，`solve_lp` 负责「系数 → 最优解」，两步分开，换题目改前者、换算法改后者，互不影响。


In [4]:
# =====================================================================
# 3. 建模与求解
# =====================================================================
from dataclasses import dataclass


@dataclass
class LpSpec:
    """把「原始表」翻译成线性规划的说明书。换题目只改这里的列名映射。

    宽表形态（一行一个决策变量，最常见）：
        方案   收益   成本   资源1  资源2  下限
        甲     1200   680    2.0    3.5    0
        乙      900   520    1.5    2.0    2

    长表形态（产品 × 资源，每行一个单耗）：
        产品   资源    单耗   资源上限
        甲     工时    2.0    800
        甲     原料    3.5    1200
    """
    var_col: str                       # 决策变量名所在列（产品/方案/项目）
    objective_col: str | None = None   # 收益列，目标里取 +
    cost_col: str | None = None        # 成本列，目标里取 -
    resource_cols: tuple = ()          # 各资源的单耗列；留空则自动推断
    capacity: dict | None = None       # {资源列名: 上限}
    lb: float = 0.0                    # 变量下界（常量）
    ub: float | None = None            # 变量上界（None = 无穷）
    lb_col: str | None = None          # 每行一个下界（如最小产量）
    long_item_col: str | None = None   # 长表：资源名所在列
    long_value_col: str | None = None  # 长表：单耗所在列
    long_cap_col: str | None = None    # 长表：资源上限所在列


def build_lp_from_table(df, spec: LpSpec, maximize=True):
    """原始表 → LP 系数（c / A_ub / b_ub / bounds）。

    内部做三件事：
    1. 长表按 (变量, 资源) pivot 成宽表，并从长表提取每种资源的上限；
    2. 目标系数 = 收益 - 成本；
    3. 资源单耗列拼成 A_ub，上限拼成 b_ub。

    返回 dict，直接 ** 传给 solve_lp 即可。
    """
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]
    spec.capacity = dict(spec.capacity or {})
    diag = []

    # ---- 长表 → 宽表 ----
    if spec.long_item_col and spec.long_value_col:
        raw = d.copy()
        if spec.long_cap_col and spec.long_cap_col in raw.columns:
            caps = (raw.assign(**{spec.long_item_col: raw[spec.long_item_col].astype("string").str.strip()})
                       .groupby(spec.long_item_col)[spec.long_cap_col].max())
            spec.capacity = {k: float(v) for k, v in caps.items()}
            diag.append(f"从长表提取资源上限：{spec.capacity}")
        d = pivot_long(raw, spec.var_col, spec.long_item_col, spec.long_value_col)
        # 单价/成本/最小产量在长表里每行重复，按变量取第一个
        for col in filter(None, [spec.objective_col, spec.cost_col, spec.lb_col]):
            if col in raw.columns:
                s = raw.groupby(spec.var_col)[col].first()
                d = d.merge(s.rename(col), left_on=spec.var_col,
                            right_index=True, how="left")
        diag.append(f"长表转宽表：{raw.shape} → {d.shape}")

    var_names = d[spec.var_col].astype(str).tolist()
    n = len(var_names)

    # ---- 目标系数 c ----
    c = np.zeros(n)
    if spec.objective_col:
        c += pd.to_numeric(d[spec.objective_col], errors="coerce").fillna(0).to_numpy()
    if spec.cost_col:
        c -= pd.to_numeric(d[spec.cost_col], errors="coerce").fillna(0).to_numpy()
    diag.append(f"目标：{'收益' if spec.objective_col else ''}"
                f"{'-成本' if spec.cost_col else ''}，{'最大化' if maximize else '最小化'}，{n} 个变量")

    # ---- 资源约束 A_ub / b_ub ----
    res_cols = list(spec.resource_cols) or [
        x for x in d.columns
        if x not in {spec.var_col, spec.objective_col, spec.cost_col, spec.lb_col}
    ]
    A_ub, b_ub = [], []
    for col in res_cols:
        if col not in d.columns:
            raise KeyError(f"资源列 {col!r} 不在表里，现有列：{list(d.columns)}")
        if col not in spec.capacity:
            raise ValueError(f"资源列 {col!r} 没给上限，请填 LpSpec.capacity")
        A_ub.append(pd.to_numeric(d[col], errors="coerce").fillna(0).to_numpy())
        b_ub.append(float(spec.capacity[col]))
    diag.append(f"{len(res_cols)} 条资源约束：{dict(zip(res_cols, b_ub))}")

    # ---- 上下界 ----
    lb = (pd.to_numeric(d[spec.lb_col], errors="coerce").fillna(spec.lb).to_numpy()
          if spec.lb_col else np.full(n, float(spec.lb)))
    ub = np.full(n, np.inf if spec.ub is None else float(spec.ub))
    bounds = list(zip(lb, ub))

    return dict(c=c, A_ub=np.array(A_ub) if A_ub else None,
                b_ub=np.array(b_ub) if b_ub else None,
                bounds=bounds, var_names=var_names, maximize=maximize,
                resource_names=res_cols, diag=diag)


def solve_lp(c, A_ub=None, b_ub=None, bounds=None, maximize=True,
             var_names=None, verbose=False):
    """调用 scipy 的 HiGHS 求解线性规划。

    最大化在内部转成最小化（c → -c），输出时再翻回来，
    所以你只需要声明 maximize，不用自己管符号。
    """
    t0 = time.time()
    c = np.asarray(c, dtype=float).ravel()
    n = len(c)
    bounds = list(bounds) if bounds is not None else [(0, None)] * n
    r = sp_opt.linprog(-c if maximize else c,
                       A_ub=np.asarray(A_ub, float) if A_ub is not None else None,
                       b_ub=np.asarray(b_ub, float) if b_ub is not None else None,
                       bounds=bounds, method="highs",
                       options={"disp": verbose})
    x = r.x if r.x is not None else np.full(n, np.nan)
    obj = float(-r.fun if maximize else r.fun) if r.fun is not None else np.nan
    return dict(x=x, objective=obj, ok=bool(r.success), status=int(r.status),
                message=str(r.message),
                var_names=list(var_names) if var_names is not None else [f"x{i}" for i in range(n)],
                maximize=maximize, elapsed=time.time() - t0)


def result_tables(res, A_ub=None, b_ub=None, resource_names=None):
    """把结果整理成三张表：决策变量 / 约束使用情况 / 摘要。"""
    t_var = pd.DataFrame({"变量": res["var_names"], "取值": np.round(res["x"], 4)})
    t_sum = pd.DataFrame([{
        "状态": "最优" if res["ok"] else "未求得最优",
        "目标值": round(res["objective"], 4),
        "方向": "最大化" if res["maximize"] else "最小化",
        "变量数": len(res["x"]), "用时(秒)": round(res["elapsed"], 3),
    }])
    if A_ub is not None and b_ub is not None:
        use = np.asarray(A_ub, float) @ res["x"]
        t_con = pd.DataFrame({
            "资源": resource_names or [f"资源{i+1}" for i in range(len(b_ub))],
            "使用量": np.round(use, 4), "上限": np.round(np.asarray(b_ub, float), 4),
            "使用率%": np.round(use / np.asarray(b_ub, float) * 100, 2),
        })
    else:
        t_con = pd.DataFrame()
    return t_var, t_con, t_sum


## 4. 示例 A：原始脏表 → LP

一份带坑的附件，全程不手动改数据。


In [5]:
# =====================================================================
# 4. 示例 A：原始脏表 → 线性规划
# =====================================================================
# 这份附件故意留了真实比赛常见的坑：
#   表头带空格、产品名带空格、资源名带空格、单耗是文本、资源上限有 999 哨兵、
#   有 1 行缺测、有 2 行完全重复。
# 下面不手动改数据，全部交给 clean_frame 处理，并打印每一步做了什么。
diag = []
raw = load_table(CFG["data_path"])
print("原始数据：", raw.shape)
print(raw.head(6).to_string())

df = clean_frame(raw, diagnostics=diag)
print("\n清洗后：", df.shape)
print(df.to_string(index=False))

print("\n--- 数据诊断（可直接写进论文的「数据预处理」一节）---")
print(pd.DataFrame(diag).to_string(index=False))

# ---- 列名映射：告诉模型哪列是什么 ----
spec = LpSpec(
    var_col="产品",              # 决策变量名
    objective_col="单价",        # 收益
    cost_col="单位成本",         # 成本 → 目标系数 = 单价 - 单位成本
    lb_col="最小产量",           # 每个产品的最小产量作为下界
    long_item_col="资源",        # 长表：资源列
    long_value_col="单耗",       # 长表：单耗列
    long_cap_col="资源上限",     # 长表：资源上限列
)
model = build_lp_from_table(df, spec, CFG["maximize"])
print("\n--- 建模信息 ---")
for line in model["diag"]:
    print(" ·", line)

res_a = solve_lp(**{k: model[k] for k in
                    ("c", "A_ub", "b_ub", "bounds", "maximize", "var_names")})
t_var_a, t_con_a, t_sum_a = result_tables(res_a, model["A_ub"], model["b_ub"],
                                          model["resource_names"])
print("\n=== 最优方案（原始表模式）===")
print(t_var_a.to_string(index=False))
print("\n=== 资源使用情况 ===")
print(t_con_a.to_string(index=False))
print("\n", t_sum_a.to_string(index=False))


原始数据： (11, 7)
      产品       资源   单耗   资源上限    单价  单位成本  最小产量
0    A型机  设备工时(h)  2.0  800.0  1200   680     4
1    A型机  原料(kg)   3.5  999.0  1200   680     4
2    A型机    人工(h)  1.2  500.0  1200   680     4
3    B型机  设备工时(h)  1.5  800.0   900   520     2
4    B型机   原料(kg)  2.0    NaN   900   520     2
5    B型机    人工(h)  0.8  500.0   900   520     2

清洗后： (9, 7)
 产品      资源  单耗   资源上限   单价  单位成本  最小产量
A型机 设备工时(h) 2.0  800.0 1200   680     4
A型机  原料(kg) 3.5  800.0 1200   680     4
A型机   人工(h) 1.2  500.0 1200   680     4
B型机 设备工时(h) 1.5  800.0  900   520     2
B型机  原料(kg) 2.0  800.0  900   520     2
B型机   人工(h) 0.8  500.0  900   520     2
C型机 设备工时(h) 3.0  800.0 1500   860     1
C型机  原料(kg) 4.5 1200.0 1500   860     1
C型机   人工(h) 1.8  500.0 1500   860     1

--- 数据诊断（可直接写进论文的「数据预处理」一节）---
    步骤                      说明
哨兵值转缺失                新增缺失 1 个
文本列去空格                  处理 2 列
 列类型识别         数值列 5 个、类别列 2 个
 去完全重复                  删除 2 行
  缺失填补 填补 ['资源上限']，数值方式 median
  清洗完成             

## 5. 示例 B：已处理矩阵 → LP

同一问题的另一种入口：系数已经算好，直接求解。


In [6]:
# =====================================================================
# 5. 示例 B：已处理矩阵 → 线性规划
# =====================================================================
# 同一个问题，假设你已经把系数整理好了（比如上一节算完、或队友给了你 Excel 里的矩阵）。
# 这时不需要 clean_frame，也不用 LpSpec，直接把数字喂进去。
# 好处是快、可控；代价是脏数据的责任在你这边——喂错了不会有任何提示。
c_b = [520, 380, 640]                       # 单价 - 单位成本
A_ub_b = [[2.0, 1.5, 3.0],                  # 设备工时单耗
          [3.5, 2.0, 4.5],                  # 原料单耗
          [1.2, 0.8, 1.8]]                  # 人工单耗
b_ub_b = [800, 1200, 500]                   # 三种资源上限
bounds_b = [(4, None), (2, None), (1, None)]  # 最小产量

res_b = solve_lp(c_b, A_ub_b, b_ub_b, bounds_b, maximize=True,
                 var_names=["A型机", "B型机", "C型机"])
t_var_b, t_con_b, t_sum_b = result_tables(res_b, A_ub_b, b_ub_b,
                                          ["设备工时(h)", "原料(kg)", "人工(h)"])
print("=== 最优方案（矩阵模式）===")
print(t_var_b.to_string(index=False))
print("\n=== 资源使用情况 ===")
print(t_con_b.to_string(index=False))
print("\n", t_sum_b.to_string(index=False))

print(f"\n两种模式目标值：原始表 {res_a['objective']:.2f} | 矩阵 {res_b['objective']:.2f}")
print("两者一致说明：清洗没有改动关键系数。若不一致，先去看上面的「数据诊断」，"
      "大概率是 999 或缺失被填补改变了单耗。")


=== 最优方案（矩阵模式）===
 变量    取值
A型机 159.4
B型机 318.8
C型机   1.0

=== 资源使用情况 ===
     资源     使用量     上限   使用率%
设备工时(h)  800.00  800.0 100.00
 原料(kg) 1200.00 1200.0 100.00
  人工(h)  448.12  500.0  89.62

 状态      目标值  方向  变量数  用时(秒)
最优 204672.0 最大化    3  0.002

两种模式目标值：原始表 204672.00 | 矩阵 204672.00
两者一致说明：清洗没有改动关键系数。若不一致，先去看上面的「数据诊断」，大概率是 999 或缺失被填补改变了单耗。


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出与画图
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL:
    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.bar(t_var_a["变量"], t_var_a["取值"], color="#4a7fd8")
    for i, v in enumerate(t_var_a["取值"]):
        ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("最优产量")
    ax.set_title(f"生产计划最优解（总利润 {res_a['objective']:.0f}）")
    fig.tight_layout()

written = export_result(
    {"结果摘要": t_sum_a, "决策变量": t_var_a, "资源使用": t_con_a,
     "清洗后数据": df, "数据诊断": pd.DataFrame(diag)},
    CFG, fig=fig, fig_name="最优产量")
print(written)


[表格] 生产计划_结果汇总.xlsx（5 个 Sheet）
[图片] 生产计划_最优产量.png
[输出]  D:\git项目\输出数据\优化_生产计划
{'主表': WindowsPath('D:/git项目/输出数据/优化_生产计划/生产计划_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_生产计划/生产计划_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_生产计划/生产计划_最优产量.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。
